# Neural Network Model

In [ ]:
# import relevant libraries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, precision_recall_fscore_support

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam

from tqdm.auto import tqdm

In [18]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Install a CUDA-enabled PyTorch build or enable GPU.")

device = torch.device("cuda")
device
torch.__version__

'2.9.0+cu126'

In training the neural network, TF-IDF of the word and character n-grams will be used as the feature to be fed throught the model. As mentioned before in the corpus analysis notebook, TF-IDF can be used as a metric to be used for classifying text authorship. For the model architecture, both shallow and deep neural network architecture will be explored. We are curious to see if a simpler architecture is more fit considering the small dataset, and the limited authors we have. Prior works have tried to use both architectures for the models [1,2,3].

The model will utilize ReLU activation functions to improve training efficiency and mitigate vanishing gradient issues [4]. In addition, dropout regularization will be introduced to reduce overfitting [5] and over reliance on specific neurons. The Adam optimizer will be used due to its adaptive learning rate and strong empirical performance across a wide range of machine learning tasks [6] Additionally, early stopping is applied to prevent overfitting by monitoring validation performance during training [7].

## Data prep and TF-IDF features

In [19]:
DATA_PATH = Path("llm_data/llm-dataset/eli5_all_llm_answers_cleaned.csv")
df = pd.read_csv(DATA_PATH)

long_df = df.melt(
    id_vars = ["q_id", "question"],
    value_vars = ["chatgpt", "deepseek", "gemini"],
    var_name = "author",
    value_name = "text",
).dropna(subset = ["text"])
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(long_df["author"])
texts = long_df["text"].astype(str).tolist()


### Data Split

In [20]:
# 70-15-15 split for train, validation, and test sets
X_train_texts, X_temp_texts, y_train, y_temp = train_test_split(
    texts, 
    labels, 
    test_size = 0.3, 
    random_state = 67, 
    stratify = labels
)
X_val_texts, X_test_texts, y_val, y_test = train_test_split(
    X_temp_texts, 
    y_temp, 
    test_size = 0.5, 
    random_state = 67, 
    stratify = y_temp
)

In [21]:
# Inspect training data and labels
train_df = pd.DataFrame({"text": X_train_texts, "label_id": y_train})
train_df["label_name"] = label_encoder.inverse_transform(train_df["label_id"])

print("Training label distribution:")
print(train_df["label_name"].value_counts())

print("\nSample training rows:")
display(train_df.head(20))

Training label distribution:
label_name
gemini      14000
deepseek    14000
chatgpt     14000
Name: count, dtype: int64

Sample training rows:


,text,label_id,label_name
0,Imagine your brain is like a super special box...,2,gemini
1,Think of it like a giant box of crayons—even i...,1,deepseek
2,Movie theaters are having a tough time for bot...,1,deepseek
3,"Penile size is mostly determined by genetics, ...",0,chatgpt
4,Maple seeds have a special wing-like part call...,1,deepseek
5,"Yes, it can affect your health because your bo...",1,deepseek
6,We still laugh at jokes we know because our br...,1,deepseek
7,AM radio is like shouting your words across a ...,1,deepseek
8,When a surgeon needs to remove part of the liv...,1,deepseek
9,Some people float on water easily because of h...,0,chatgpt


In [22]:
# Inspect validation data and labels
val_df = pd.DataFrame({"text": X_val_texts, "label_id": y_val})
val_df["label_name"] = label_encoder.inverse_transform(val_df["label_id"])

print("Validation label distribution:")
print(val_df["label_name"].value_counts())

print("\nSample validation rows:")
display(val_df.sample(min(5, len(val_df)), random_state=67))

print(val_df.sample(min(5, len(val_df)), random_state=67))

Validation label distribution:
label_name
gemini      3000
deepseek    3000
chatgpt     3000
Name: count, dtype: int64

Sample validation rows:


,text,label_id,label_name
1073,"Okay, imagine you have a magic powder that mak...",0,chatgpt
7698,Hangnails are little pieces of skin that get s...,0,chatgpt
3879,"Okay! So, your eyes have tiny helpers called c...",0,chatgpt
1620,"In documentaries, people's faces are blurred t...",1,deepseek
8023,That's a great question! Think of your brain l...,1,deepseek


                                                   text  label_id label_name
1073  Okay, imagine you have a magic powder that mak...         0    chatgpt
7698  Hangnails are little pieces of skin that get s...         0    chatgpt
3879  Okay! So, your eyes have tiny helpers called c...         0    chatgpt
1620  In documentaries, people's faces are blurred t...         1   deepseek
8023  That's a great question! Think of your brain l...         1   deepseek


### Getting the n-grams

In [23]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range = (2, 3),  # word bigram and trigram 
    min_df = 10, # increase min_df to reduce noise and dimensionality
    max_features = 6767, # limit for computational efficiency
    dtype = np.float32,
    )

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range = (3, 4), # char 3-grams to 4-grams
    min_df = 10, # increase min_df to reduce noise and dimensionality
    max_features = 6767,  # limit for computational efficiency
    dtype = np.float32,
    )

X_train_word = word_vectorizer.fit_transform(X_train_texts)
X_val_word = word_vectorizer.transform(X_val_texts)
X_test_word = word_vectorizer.transform(X_test_texts)

X_train_char = char_vectorizer.fit_transform(X_train_texts)
X_val_char = char_vectorizer.transform(X_val_texts)
X_test_char = char_vectorizer.transform(X_test_texts)

num_features_word = X_train_word.shape[1]
num_features_char = X_train_char.shape[1]
num_classes = len(label_encoder.classes_)

## Grid search for hidden layer sizes

### Globals


In [38]:
start_neurons = 32
max_neurons = 512
max_layers = 3
lr_search_space = [1e-4, 5e-5, 2e-4]

best_config_by_feature = {}
best_val_acc_by_feature = {}
results_by_feature = {}
histories_by_feature = {}


### Helper Functions

In [ ]:
class LLMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_units, num_classes, dropout_rate = 0.3):
        super().__init__()
        layers_list = []
        prev_dim = input_dim
        for units in hidden_units:
            layers_list.append(nn.Linear(prev_dim, units))
            layers_list.append(nn.ReLU())
            layers_list.append(nn.Dropout(dropout_rate))
            prev_dim = units
        layers_list.append(nn.Linear(prev_dim, num_classes))
        self.model = nn.Sequential(*layers_list)

    def forward(self, x):
        return self.model(x)

def make_dataloader(features, labels, batch_size, shuffle = False):
    features_tensor = torch.tensor(features, dtype = torch.float32)
    labels_tensor = torch.tensor(labels, dtype = torch.long)
    dataset = TensorDataset(features_tensor, labels_tensor)
    return DataLoader(dataset, batch_size = batch_size, shuffle = shuffle)

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = loss_fn(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_x.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            loss = loss_fn(logits, batch_y)
            total_loss += loss.item() * batch_x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_x.size(0)
    return total_loss / total, correct / total

def final_evaluation(model, loader, device, target_names=None):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            logits = model(batch_x)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(batch_y.numpy())
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets,
        all_preds,
        average="weighted",
        zero_division=0,
    )
    correct = np.sum(np.array(all_targets) == np.array(all_preds))
    accuracy = correct / len(all_targets) if all_targets else 0.0
    report = classification_report(
        all_targets,
        all_preds,
        target_names=target_names,
        digits=4,
        zero_division=0,
    )
    metrics = {
        "accuracy": accuracy,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
        "report": report,
    }
    return metrics

def fit_model(model, train_loader, val_loader, epochs, lr, patience, device):
    optimizer = Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best_state = None
    best_val_loss = float("inf")
    patience_left = patience
    history = []
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })
        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left == 0:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return history

def get_best_val_loss(history):
    if not history:
        return None, None, None
    val_losses = [h["val_loss"] for h in history]
    val_accs = [h["val_acc"] for h in history]
    best_idx = int(np.argmin(val_losses))
    return history[best_idx]["epoch"], val_losses[best_idx], val_accs[best_idx]

def get_model_checkpoint_path(feature):
    return Path("models") / f"best_model_{feature}.pt"

def load_grid_search_history(path=None):
    input_path = Path(path) if path is not None else Path("models") / "grid_search_histories.json"
    if not input_path.exists():
        return {}
    with input_path.open("r", encoding="utf-8") as handle:
        raw = json.load(handle)
    normalized = {}
    for feature_name, histories in raw.items():
        normalized[feature_name] = [
            {
                "hidden_units": tuple(item.get("hidden_units", [])),
                "lr": item.get("lr"),
                "history": item.get("history", []),
            }
            for item in histories
        ]
    return normalized

def save_best_model(feature, model, best_cfg, label_encoder, vectorizer, input_dim, num_classes, dropout_rate=0.5, path=None):
    checkpoint_path = Path(path) if path is not None else get_model_checkpoint_path(feature)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    label_classes = label_encoder.classes_.tolist() if hasattr(label_encoder, "classes_") else None
    vectorizer_params = vectorizer.get_params() if hasattr(vectorizer, "get_params") else None
    vectorizer_vocab = getattr(vectorizer, "vocabulary_", None)
    vectorizer_idf = vectorizer.idf_.tolist() if hasattr(vectorizer, "idf_") else None
    checkpoint = {
        "feature": feature,
        "input_dim": input_dim,
        "num_classes": num_classes,
        "hidden_units": best_cfg["hidden_units"],
        "lr": best_cfg["lr"],
        "dropout_rate": dropout_rate,
        "state_dict": model.state_dict(),
        "label_classes": label_classes,
        "vectorizer_params": vectorizer_params,
        "vectorizer_vocab": vectorizer_vocab,
        "vectorizer_idf": vectorizer_idf,
    }
    torch.save(checkpoint, checkpoint_path)
    return checkpoint_path

def load_best_model(feature, device, path=None):
    checkpoint_path = Path(path) if path is not None else get_model_checkpoint_path(feature)
    if not checkpoint_path.exists():
        return None, None
    try:
        if hasattr(torch, "serialization") and hasattr(torch.serialization, "safe_globals"):
            with torch.serialization.safe_globals([LabelEncoder, TfidfVectorizer]):
                checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        else:
            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    except Exception as exc:
        raise RuntimeError(
            f"Failed to load checkpoint at {checkpoint_path}. Delete it and retrain if it was saved with older format."
        ) from exc

    model = LLMClassifier(
        input_dim = checkpoint["input_dim"],
        hidden_units = checkpoint["hidden_units"],
        num_classes = checkpoint["num_classes"],
        dropout_rate = checkpoint.get("dropout_rate", 0.5),
    ).to(device)

    model.load_state_dict(checkpoint["state_dict"])

    return model, checkpoint

def run_grid_search(feature_name, cfg):
    best_config = None
    best_val_acc = 0.0
    results = []
    histories = []
    prev_best_val_loss = None
    neurons = start_neurons

    while neurons <= max_neurons:
        best_val_loss_size = None
        for layers in range(1, max_layers + 1):
            hidden_units = (neurons,) * layers
            for lr in lr_search_space:
                print(f"\n[{feature_name}] Training model with hidden_units={hidden_units}, lr={lr}")
                model = LLMClassifier(
                    input_dim=cfg["input_dim"],
                    hidden_units=hidden_units,
                    num_classes=num_classes,
                    dropout_rate=0.5,
                ).to(device)

                history = fit_model(
                    model,
                    cfg["train_loader"],
                    cfg["val_loader"],
                    epochs = 40,
                    lr = lr,
                    patience = 2,
                    device=device,
                )
                val_acc = history[-1]["val_acc"] if history else 0.0
                results.append({"hidden_units": hidden_units, "lr": lr, "val_acc": val_acc})
                histories.append({"hidden_units": hidden_units, "lr": lr, "history": history})

                best_epoch, best_epoch_val_loss, best_epoch_val_acc = get_best_val_loss(history)
                if best_epoch is not None:
                    print(
                        f"[{feature_name}] Best val loss for {hidden_units}, lr={lr}: "
                        f"epoch {best_epoch} (val_loss={best_epoch_val_loss:.4f})"
                    )
                    if best_epoch_val_acc > best_val_acc:
                        best_val_acc = best_epoch_val_acc
                        best_config = {"hidden_units": hidden_units, "lr": lr}
                    best_val_loss_size = (
                        best_epoch_val_loss
                        if best_val_loss_size is None
                        else min(best_val_loss_size, best_epoch_val_loss)
                    )

        neurons *= 2

    return best_config, best_val_acc, results, histories

# Evaluate best models on test set (word + char)
def train_best_for_feature(feature, train_loader, val_loader, test_loader):
    best_cfg = best_config_by_feature.get(feature)
    if best_cfg is None:
        raise RuntimeError(f"best_config not found for feature: {feature}")
    if feature == "word":
        input_dim = X_train_word.shape[1]
        train_loader = train_loader
        val_loader = val_loader
        test_loader = test_loader
        vectorizer = word_vectorizer
    else:
        input_dim = X_train_char.shape[1]
        train_loader = train_loader
        val_loader = val_loader
        test_loader = test_loader
        vectorizer = char_vectorizer

    model = LLMClassifier(
        input_dim = input_dim,
        hidden_units = best_cfg["hidden_units"],
        num_classes = num_classes,
        dropout_rate = 0.5,
    ).to(device)

    fit_model(
        model,
        train_loader,
        val_loader,
        epochs = 40,
        lr = best_cfg["lr"],
        patience = 2,
        device = device,
    )

    save_best_model(
        feature = feature,
        model = model,
        best_cfg = best_cfg,
        label_encoder = label_encoder,
        vectorizer = vectorizer,
        input_dim = input_dim,
        num_classes = num_classes,
        dropout_rate = 0.5,
    )
    
    test_loss, test_acc = evaluate(model, test_loader, nn.CrossEntropyLoss(), device)
    
    return test_loss, test_acc

### Data Prep and Loader for MiniBatch Gradient Descent

In [26]:
X_train_word_dense = X_train_word.toarray().astype(np.float32)
X_val_word_dense = X_val_word.toarray().astype(np.float32)
X_test_word_dense = X_test_word.toarray().astype(np.float32)

X_train_char_dense = X_train_char.toarray().astype(np.float32)
X_val_char_dense = X_val_char.toarray().astype(np.float32)
X_test_char_dense = X_test_char.toarray().astype(np.float32)

train_loader_word = make_dataloader(X_train_word_dense, y_train, batch_size = 32, shuffle = True)
val_loader_word = make_dataloader(X_val_word_dense, y_val, batch_size = 32)
test_loader_word = make_dataloader(X_test_word_dense, y_test, batch_size = 32)

train_loader_char = make_dataloader(X_train_char_dense, y_train, batch_size = 32, shuffle = True)
val_loader_char = make_dataloader(X_val_char_dense, y_val, batch_size = 32)
test_loader_char = make_dataloader(X_test_char_dense, y_test, batch_size = 32)

### Variables for Grid Search

In [27]:
feature_configs = {
    "word": {
        "input_dim": X_train_word.shape[1],
        "train_loader": train_loader_word,
        "val_loader": val_loader_word,
    },
    "char": {
        "input_dim": X_train_char.shape[1],
        "train_loader": train_loader_char,
        "val_loader": val_loader_char,
    },
}

### Grid Search Run

In [12]:
for feature_name, cfg in feature_configs.items():
    best_cfg, best_acc, results, histories = run_grid_search(feature_name, cfg)
    best_config_by_feature[feature_name] = best_cfg
    best_val_acc_by_feature[feature_name] = best_acc
    results_by_feature[feature_name] = results
    histories_by_feature[feature_name] = histories

best_config_by_feature, best_val_acc_by_feature


[word] Training model with hidden_units=(16,), lr=0.0001
Epoch 01 | train_loss=1.0530 train_acc=0.4961 val_loss=0.9701 val_acc=0.8853
Epoch 01 | train_loss=1.0530 train_acc=0.4961 val_loss=0.9701 val_acc=0.8853
Epoch 02 | train_loss=0.8739 train_acc=0.7930 val_loss=0.7628 val_acc=0.9049
Epoch 02 | train_loss=0.8739 train_acc=0.7930 val_loss=0.7628 val_acc=0.9049
Epoch 03 | train_loss=0.6988 train_acc=0.8218 val_loss=0.5934 val_acc=0.9143
Epoch 03 | train_loss=0.6988 train_acc=0.8218 val_loss=0.5934 val_acc=0.9143
Epoch 04 | train_loss=0.5752 train_acc=0.8437 val_loss=0.4750 val_acc=0.9204
Epoch 04 | train_loss=0.5752 train_acc=0.8437 val_loss=0.4750 val_acc=0.9204
Epoch 05 | train_loss=0.4914 train_acc=0.8570 val_loss=0.3932 val_acc=0.9264
Epoch 05 | train_loss=0.4914 train_acc=0.8570 val_loss=0.3932 val_acc=0.9264
Epoch 06 | train_loss=0.4285 train_acc=0.8734 val_loss=0.3347 val_acc=0.9292
Epoch 06 | train_loss=0.4285 train_acc=0.8734 val_loss=0.3347 val_acc=0.9292
Epoch 07 | train_l

({'word': {'hidden_units': (128,), 'lr': 5e-05},
  'char': {'hidden_units': (512,), 'lr': 5e-05}},
 {'word': 0.9482222222222222, 'char': 0.9845555555555555})

## Train with the best configuration, and Test with test dataset

### Best Word n-gram Model

In [ ]:
word_test_results = {}
feature = "word"
model, checkpoint = load_best_model(feature, device)

if model is None:
    print(f"No saved model found for {feature}. Training and saving...")
    test_loss, test_acc = train_best_for_feature(feature, train_loader_word, val_loader_word, test_loader_word)
    model, _ = load_best_model(feature, device)

word_metrics = final_evaluation(
    model,
    test_loader_word,
    device,
    target_names=label_encoder.classes_.tolist(),
)
print(word_metrics["report"])


Loaded saved model for word from models\best_model_word.pt


{'word': {'test_loss': 0.14002705946233537, 'test_acc': 0.9474444444444444}}

### Best Char n-gram Model

In [ ]:
feature = "char"
model, checkpoint = load_best_model(feature, device)

if model is None:
    print(f"No saved model found for {feature}. Training and saving...")
    test_loss, test_acc = train_best_for_feature(feature, train_loader_char, val_loader_char, test_loader_char)
    model, _ = load_best_model(feature, device)

char_metrics = final_evaluation(
    model,
    test_loader_char,
    device,
    target_names=label_encoder.classes_.tolist(),
)
print(char_metrics["report"])


              precision    recall  f1-score   support

     chatgpt     0.9803    0.9807    0.9805      3000
    deepseek     0.9810    0.9787    0.9798      3000
      gemini     0.9870    0.9890    0.9880      3000

    accuracy                         0.9828      9000
   macro avg     0.9828    0.9828    0.9828      9000
weighted avg     0.9828    0.9828    0.9828      9000



## Graphs of Loss and Accuracy

In [40]:
import ipywidgets as widgets
from IPython.display import display

histories_by_feature_ui = load_grid_search_history()
if not histories_by_feature_ui:
    histories_by_feature_ui = histories_by_feature

feature_dropdown = widgets.Dropdown(options=["word", "char"], description="Feature:")
lr_dropdown = widgets.Dropdown(description="LR:")
hidden_dropdown = widgets.Dropdown(description="Neuron:")

def _history_options(feature):
    histories = histories_by_feature_ui.get(feature, [])
    if not histories:
        return sorted(lr_search_space), _default_hidden_units()
    lrs = sorted({item.get("lr") for item in histories if item.get("lr") is not None})
    hidden_units_set = {
        tuple(item.get("hidden_units", ()))
        for item in histories
        if item.get("hidden_units")
    }
    hidden_units_list = sorted(hidden_units_set, key=lambda x: (len(x), x[0] if x else 0))
    return lrs, hidden_units_list

def _default_hidden_units():
    hidden_units_list = []
    neurons = start_neurons
    while neurons <= max_neurons:
        for layers in range(1, max_layers + 1):
            hidden_units_list.append(tuple([neurons] * layers))
        neurons *= 2
    return hidden_units_list

def update_options(*args):
    lrs, hidden_units_list = _history_options(feature_dropdown.value)
    if not lrs:
        lrs = sorted(lr_search_space)
    if not hidden_units_list:
        hidden_units_list = _default_hidden_units()
    lr_dropdown.options = lrs
    hidden_dropdown.options = hidden_units_list
    if lrs:
        lr_dropdown.value = lrs[0]
    if hidden_units_list:
        hidden_dropdown.value = hidden_units_list[0]

feature_dropdown.observe(update_options, names="value")
update_options()

def plot_history(feature, lr, hidden_units):
    histories = histories_by_feature_ui.get(feature, [])
    match = None
    for item in histories:
        if item["lr"] == lr and item["hidden_units"] == hidden_units:
            match = item
            break
    if match is None or not match["history"]:
        print("No history found for this configuration.")
        return
    history = match["history"]
    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    train_acc = [h["train_acc"] for h in history]
    val_acc = [h["val_acc"] for h in history]

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_loss, label="train")
    plt.plot(epochs, val_loss, linestyle="--", label="val")
    plt.title("Loss per epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.ylim(0.0, 1.5)
    plt.legend(fontsize=8)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_acc, label="train")
    plt.plot(epochs, val_acc, linestyle="--", label="val")
    plt.title("Accuracy per epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.ylim(0.0, 1.0)
    plt.legend(fontsize=8)

    plt.suptitle(f"{feature} n-grams | lr={lr}, neurons={hidden_units}")
    plt.tight_layout()
    plt.show()

ui = widgets.HBox([feature_dropdown, lr_dropdown, hidden_dropdown])
out = widgets.interactive_output(
    plot_history,
    {"feature": feature_dropdown, "lr": lr_dropdown, "hidden_units": hidden_dropdown},
)
display(ui, out)

NameError: name 'load_grid_search_history' is not defined

[1]
Modupe, A., Celik, T., Marivate, V., & Olugbara, O. O. (2022). Post-authorship attribution using regularized deep neural network. Applied Sciences, 12(15), 7518.

[2]
Saha, N., Das, P., & Saha, H. N. (2018). Authorship attribution of short texts using multi-layer perceptron. International Journal of Applied Pattern Recognition, 5(3), 251-259.

[3]
Sari, Y., Vlachos, A., & Stevenson, M. (2017, April). Continuous n-gram representations for authorship attribution. In Proceedings of the 15th conference of the European chapter of the association for computational linguistics: Volume 2, short papers (pp. 267-273).

[4]
GeeksforGeeks. (2025, March 11). ReLU activation function in deep learning. https://www.geeksforgeeks.org/deep-learning/relu-activation-function-in-deep-learning/

[5]
Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). Dropout: a simple way to prevent neural networks from overfitting. The journal of machine learning research, 15(1), 1929-1958.

[6]
Kingma, D. P., & Ba, J. (2014). Adam: A method for stochastic optimization. arXiv preprint arXiv:1412.6980.

[7]
GeeksforGeeks. (2025, August 28). Regularization by early stopping. https://www.geeksforgeeks.org/machine-learning/regularization-by-early-stopping/